# Delux AI Zero
- Solving Lux AI season 1 challenge by using MuZero implementation and AlphaStar Encoding scheme.

## Current State
- Please be aware that the code is not tested yet, there is still pending components need to be done. Because of homework and school exams, I have to stop coding for this project. The MCTS tree is not really working yet. However, I will still share it anyway and hope someone can get some inspiration here.

## Notes about MuZero

Inputs:
- current and past game states

Model Functions:
- Representation function h(observation) -> hidden state
- Prediction function f(hidden state) -> policy and value
- Dynamic function g(hidden state, action) -> reward and next hidden state

Things to predict:

- policy 
- value 
- reward

# Libraries

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from lux.game import Game
from lux.game_map import Cell, RESOURCE_TYPES, Position
from lux.constants import Constants
from lux.game_constants import GAME_CONSTANTS
from lux import annotate
from tqdm.notebook import tqdm_notebook
from collections import deque
from tqdm.notebook import tqdm
import math
import sys
import copy
import random
import time

In [ ]:
!node --version
!pip install kaggle-environments -U
from kaggle_environments import make

In [ ]:
!cp -r ../input/lux-ai-2021/* .

# Experiment

In [ ]:
# game_state = None
def agent(observation, configuration):
    global game_state

    ### Do not edit ###
    if observation["step"] == 0:
        game_state = Game()
        game_state._initialize(observation["updates"])
        game_state._update(observation["updates"][2:])
        game_state.id = observation.player
    else:
        game_state._update(observation["updates"])
    
    actions = []

    ### AI Code goes down here! ### 
    player = game_state.players[observation.player]
    opponent = game_state.players[(observation.player + 1) % 2]
    width, height = game_state.map.width, game_state.map.height
    
    return actions

env = make("lux_ai_2021", configuration={"seed": random.randint(0, 100000000), "loglevel": 2}, debug=True)
steps = env.run([agent, "simple_agent"])
env.render(mode="ipython", width=1200, height=800)

# Data Input Pipeline
Let's copy mini AlphaStar, divide the inputs into 3 categories (map, unit, scalar) and embedding + concat them into tensors before feed into the core.

### Unit Encoder
Unit info is embedded inside one hot encoded vector with dimension 23 + 1 (the extra one dim ease the multi head attention).

These vectors then feed into the self attention layer (4 heads) and fed into some fc and conv 1d layers to get each unit's embedding, and general embedding of units. (if i am not confused)


In [ ]:
def dec_to_bits(value, bits):
    return np.unpackbits(np.bitwise_and(np.array(2**bits-1, dtype=np.uint8), np.array([value], dtype=np.uint8)))[-bits:]

def bits_to_dec(bits):
    # change from the bits to dec values.
    bits = tf.squeeze(bits) if isinstance(bits, tf.Tensor) and bits.ndim > 1 else bits
    l = len(bits)
    v = 0
    g = 1
    for i in range(l - 1, -1, -1):               
        v += g if bits[i] == 1 else 0
        g *= 2
    return v

def to_one_hot(tensor):
    return tf.one_hot(tf.argmax(tensor, axis=-1), tensor.shape[-1])

def scaled_dot_product_attention(q, k, v, mask=None):
    matmul_qk = tf.matmul(q, k, transpose_b=True)
    dk = tf.cast(tf.shape(k)[-1], tf.float32)
    scaled_attention_logits = matmul_qk / tf.math.sqrt(dk)
    
    if mask is not None:
        scaled_attention_logits += (mask * -1e9)
    
    attention_weights = tf.nn.softmax(scaled_attention_logits, axis=-1)
    output = tf.matmul(attention_weights, v)
    
    return output, attention_weights

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, num_heads, d_input):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_input = d_input
        
        assert d_input % num_heads == 0
        
        self.depth = d_input // num_heads
        
        self.wq = tf.keras.layers.Dense(d_input)
        self.wk = tf.keras.layers.Dense(d_input)
        self.wv = tf.keras.layers.Dense(d_input)
        
        self.dense = tf.keras.layers.Dense(d_input, activation=tf.nn.elu)
    
    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(x, perm=[0, 2, 1, 3])
    
    def call(self, input_tensors):
        batch_size = input_tensors.shape[0]
        
        q = self.wq(input_tensors) # (batch_size, input_length, d_input)
        k = self.wk(input_tensors)
        v = self.wv(input_tensors)
        
        q = self.split_heads(q, batch_size) # (batch_size, num_heads, input_length, depth)
        k = self.split_heads(k, batch_size)
        v = self.split_heads(v, batch_size)
        
        scaled_attention, attention_weights = scaled_dot_product_attention(q, k, v)
        transposed_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(transposed_attention, (batch_size, -1, self.d_input))
        
        output = self.dense(concat_attention)
        
        return output, attention_weights

In [ ]:
class UnitEncoder(tf.keras.layers.Layer):
    def __init__(self, is_opponent=False):
        super(UnitEncoder, self).__init__()
        self.mha = MultiHeadAttention(num_heads=4, d_input=24)
        self.conv = tf.keras.layers.Conv1D(16, 1)
        self.dense = tf.keras.layers.Dense(16, activation=tf.nn.elu)
        self.is_opponent = is_opponent
    
    def get_units(self, game_state):
        units = []
        padding = (32 - game_state.map_height) // 2
        for player in game_state.players:
            for unit in player.units:
                # the arrange of the data are weird but i put it this way so that transformer can work when splitting, hopefully
                unit_info = np.zeros(24, dtype=np.float32)
                unit_info[0] = unit.team % 2 if not self.is_opponent else (unit.team + 1) % 2 
                unit_info[6] = unit.is_worker()
                # unit_info[12] = is_city
                unit_info[1:6] = dec_to_bits(unit.pos.x + padding, 5)
                unit_info[7:12] = dec_to_bits(unit.pos.y + padding, 5)
                unit_info[13:17] = dec_to_bits(unit.cooldown, 4)
                unit_info[17:23] = dec_to_bits(int(math.sqrt(min(unit.get_cargo_space_left(), 2000)))+1, 6)
                units.append(unit_info)
            
            for city in player.cities.values():
                for city_tile in city.citytiles:
                    unit_info = np.zeros(24, dtype=np.float32)
                    unit_info[0] = city_tile.team % 2 if not self.is_opponent else (city_tile.team + 1) % 2
                    # unit_info[6] = unit.is_worker()
                    unit_info[12] = True # is city
                    unit_info[1:6] = dec_to_bits(city_tile.pos.x + padding, 5)
                    unit_info[7:12] = dec_to_bits(city_tile.pos.y + padding, 5)
                    unit_info[13:17] = dec_to_bits(city_tile.cooldown, 4)
                    # unit_info[17:23] is about cargo 
                    units.append(unit_info)
                    
        return units
    
    def __call__(self, game_state):
        units = self.get_units(game_state)
        units = units if isinstance(units, tf.Tensor) else tf.constant(units)
        units = tf.expand_dims(units, axis=0) if (units.ndim == 2) else units
        
        output, attention = self.mha(units)
        embedded_units = self.conv(output)
        unit_embedding = tf.reduce_mean(output, axis=1)
        unit_embedding = self.dense(unit_embedding)
        
        # embedded_units is the embedding of each unit's information, shape: (BS, num_units, unit_emb_size)
        # unit_embedding is the general embedding (mean) of all units' information, shape: (BS, unit_emb_size)
        return unit_embedding, embedded_units, units

In [ ]:
# Unit Encoder Test
unitEncoder = UnitEncoder()
unit_embedding, embedded_units, raw_units = unitEncoder(game_state)
print(embedded_units.shape)
print(unit_embedding.shape)
print(raw_units.shape)
raw_units

## Spatial encoder
Get the map and reshape it to (4, 32, 32)

Concat the scatter map (which is the pos info from embedded_units)

Output (1, 16, 6, 6) state map

In [ ]:
class SpatialEncoder(tf.keras.layers.Layer):
    def __init__(self, is_opponent=False):
        super(SpatialEncoder, self).__init__()
        self.project = tf.keras.layers.Conv2D(32, 1, activation=tf.nn.elu)
        self.conv1 = tf.keras.layers.Conv2D(4, 5, activation=tf.nn.elu, padding='same')
        self.conv2 = tf.keras.layers.Conv2D(4, 5, activation=tf.nn.elu, padding='same')
        self.conv3 = tf.keras.layers.Conv2D(2, 5, activation=tf.nn.elu, padding='same')
        self.dense = tf.keras.layers.Dense(64, activation=tf.nn.elu)
        self.flatten = tf.keras.layers.Flatten()
        self.conv_scatter = tf.keras.layers.Conv1D(4, 1, activation=tf.nn.elu)
        self.is_opponent = is_opponent
        
    
    def scatter(self, embedded_units, units_x_y):
        # reduced_embedded_units, shape: (BS, num_units, 32)
        reduced_embedded_units = tf.nn.elu(self.conv_scatter(embedded_units))
        
        batch_size = reduced_embedded_units.shape[0]
        num_units = reduced_embedded_units.shape[1]
        
        scatter_map = np.zeros(shape=(batch_size, 4, 32, 32))
        
        for batch in range(batch_size):
            for unit in range(num_units):
                x = units_x_y[batch, unit, :5]
                y = units_x_y[batch, unit, 5:]
                x = bits_to_dec(x)
                y = bits_to_dec(y)
                scatter_map[batch, :, y, x] += reduced_embedded_units[batch, unit, :].numpy()
        
        scatter_map = tf.constant(scatter_map, dtype=tf.float32)
        return scatter_map
    
    def get_map(self, game_state):
        map_size = game_state.map_height
        padding = (32 - map_size) // 2
        map_ = np.zeros((4, 32, 32), np.float32)
        for y in range(map_size):
            for x in range(map_size):
                cell = game_state.map.get_cell(x, y)
                if not cell.has_resource():
                    if cell.road != 0.0:
                        map_[0, padding+y, padding+x] = cell.road
                    continue
                if(cell.resource.type == 'wood'):
                    map_[1, padding+y, padding+x] += cell.resource.amount
                elif(cell.resource.type == 'coal'):
                    map_[2, padding+y, padding+x] += cell.resource.amount
                elif(cell.resource.type == 'uranium'):
                    map_[3, padding+y, padding+x] += cell.resource.amount
        
        return map_
                    
    def __call__(self, game_state, embedded_units, raw_units):
        game_map = self.get_map(game_state)
        game_map = game_map if isinstance(game_map, tf.Tensor) else tf.constant(game_map)
        game_map = tf.expand_dims(game_map, axis=0) if (game_map.ndim == 3) else game_map
        
        units_x_y = tf.concat([raw_units[:, :, 1:6], raw_units[:, :, 7:12]], axis=-1)
        
        scatter_map = self.scatter(embedded_units, units_x_y)
        game_map = tf.concat([game_map, scatter_map], axis=1)
        game_map = tf.transpose(game_map, perm=[0, 2, 3, 1])
        x = self.project(game_map)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = tf.transpose(x, perm=[0, 3, 1, 2])
        map_skip = x 
        x = self.flatten(x)
        x = self.dense(x)
        return x, map_skip

In [ ]:
# Spatial Encoder Test
unitEncoder = UnitEncoder()
unit_embedding, embedded_units, raw_units = unitEncoder(game_state)

spatialEncoder = SpatialEncoder()
vec, map_ = spatialEncoder(game_state, embedded_units, raw_units)
print(vec.shape)
print(map_.shape)

channels = 0
gs = []
fig, axs = plt.subplots(1, 2)
gs.append(sns.heatmap(map_[0, channels], ax=axs[0], cbar=False))
gs.append(sns.heatmap(map_[0, channels+1], ax=axs[1], cbar=False))

for g in gs:
    g.set_xlabel("")
    g.set_ylabel("")
    g.set_xticks([])
    g.set_yticks([])
    g.set(adjustable='box', aspect='equal')

## Scalar Encoder

In [ ]:
class ScalarEncoder(tf.keras.layers.Layer):
    def __init__(self, is_opponent=False):
        super(ScalarEncoder, self).__init__()
        self.dense1 = tf.keras.layers.Dense(32, activation=tf.nn.elu)
        self.dense2 = tf.keras.layers.Dense(16, activation=tf.nn.elu)
        self.is_opponent = is_opponent
    
    def get_stats(self, game_state):
        player = game_state.players[0]
        opponent = game_state.players[1]
        if self.is_opponent:
            player, opponent = opponent, player
        stats = np.zeros(9)
        stats[0] = math.cos(math.pi * (game_state.turn % 40) / 40)
        for city in player.cities.values():
            stats[1] += city.fuel
        for city in opponent.cities.values():
            stats[2] += city.fuel
        stats[3:5] = player.research_points, opponent.research_points
        stats[5:] = player.researched_coal(), player.researched_uranium(), opponent.researched_coal(), opponent.researched_uranium()
        
        return stats
    
    def __call__(self, game_state):
        game_stats = self.get_stats(game_state)
        game_stats = game_stats if isinstance(game_stats, tf.Tensor) else tf.constant(game_stats)
        game_stats = tf.expand_dims(game_stats, axis=0) if (game_stats.ndim == 1) else game_stats
        
        x = self.dense1(game_stats)
        x = self.dense2(x)
        
        return x

In [ ]:
# Scalar Encoder Test
scalarEncoder = ScalarEncoder()
scalarEncoder(game_state)

# Model Building

## Monte Carlo Tree Search (MCTS)

In [ ]:
class Node:
    def __init__(self, state, encoded_policy, reward):
        self.state = state
        self.visit_count = 0
        # Q values
        self.Q = 0
        # Prior probability of choosing this action from neural network
        encoded_policy = encoded_policy[0] if encoded_policy.ndim == 2 else encoded_policy
        self.P = encoded_policy
        # Reward
        reward = reward.numpy()[0,0] if reward.ndim == 2 else reward
        self.R = reward           
        self.children = [None]*32
    
    def __repr__(self):
        return f"Node object\nvisit counts : {self.visit_count}\nQ value : {self.Q}\nR value : {self.R}\n"
        
class MCTS:
    def __init__(self, root_state, root_encoded_policy, reward):
        self.root = Node(root_state, root_encoded_policy, reward)
    
    def add_policy(self, node, policy):
        node.P = policy
    
    def get_ucb_score(self, parent_node):
        #  c1 = 1.25 and c2 = 19652
        ucb_score = [0]*32
        c_1, c_2 = 1.25, 100
        for i in range(32):
            if parent_node.children[i] is None:
                ucb_score[i] = c_1 + np.log((parent_node.visit_count + c_2 + 1)/c_2)
                ucb_score[i] *= parent_node.P[i].numpy() * np.sqrt(parent_node.visit_count)
            else:
                ucb_score[i] = c_1 + np.log((parent_node.visit_count + c_2 + 1)/c_2)
                ucb_score[i] *= parent_node.P[i].numpy() * (np.sqrt(parent_node.visit_count))/(1+parent_node.children[i].visit_count)
                ucb_score[i] += parent_node.children[i].Q
        
        return tf.constant(ucb_score)
        
    def select(self):
        selected_node = self.root
        path = [selected_node]
        while True:
            ucb_score = self.get_ucb_score(selected_node)
            #print(ucb_score)
            selected_policy_index = np.argmax(ucb_score)
            
            if selected_node.children[selected_policy_index] is None:
                return path, selected_policy_index
            
            selected_node = selected_node.children[selected_policy_index]
            path.append(selected_node)
    
    def expand(self, node, next_node_hidden_state, node_reward):
#         node.children[]
        return

## Model class
Representation model

remove residual block and use conv for now

Inputs: game state map
Outputs: list of actions

since the muzero is designed for singe agent game, we have to improvise a bit.

Methods:
- Self play
- Learn from replay buffer

In [ ]:
class Delux(tf.keras.Model):
    def __init__(self, simulation_limit=5, discount_factor=0.997, is_opponent=False, K=5):
        super(Delux, self).__init__()
        self.prev_hidden_state = None
        self.simulation_limit = simulation_limit
        self.discount_factor = discount_factor
        self.is_opponent = is_opponent
        
        self.memory = deque(maxlen=100)
        self.buffer = []
        self.unroll_step = K
        
        self.loss_cce = tf.keras.losses.CategoricalCrossentropy()
        self.loss_mse = tf.keras.losses.MeanSquaredError()
        self.optimizer = tf.keras.optimizers.Adam(0.05)
        
        self.unitEncoder = UnitEncoder(is_opponent=is_opponent)
        self.spatialEncoder = SpatialEncoder(is_opponent=is_opponent)
        self.scalarEncoder = ScalarEncoder(is_opponent=is_opponent)
        
        # Prediction Function
        ## TODO: embedded_map should be use in this section, how?
        f_input = tf.keras.Input(shape=(96))
        f = tf.keras.layers.Dense(48, activation=tf.nn.elu)(f_input)
        v = tf.keras.layers.Dense(8, activation=tf.nn.elu)(f)
        v = tf.keras.layers.Dense(1)(v)
        
        p = tf.keras.layers.Dense(32, activation=tf.nn.elu)(f)
        p = tf.keras.layers.Dense(32, activation=tf.nn.softmax)(p)
        self.Prediction = tf.keras.Model([f_input], [p, v], name='Prediction Model')
        
        # Dynamic Function
        g_hidden_state = tf.keras.Input(shape=(96))
        g_hidden_actions = tf.keras.Input(shape=(32))
        g_input = tf.concat([g_hidden_state, g_hidden_actions], axis=-1)
        g = tf.keras.layers.Dense(96, activation=tf.nn.elu)(g_input)
        g = tf.keras.layers.Dense(96, activation=tf.nn.elu)(g_input)
        r = tf.keras.layers.Dense(32, activation=tf.nn.elu)(g)
        r = tf.keras.layers.Dense(8, activation=tf.nn.elu)(r)
        r = tf.keras.layers.Dense(1)(r)
        self.Dynamic = tf.keras.Model([g_hidden_state, g_hidden_actions], [g, r], name='Dynamic Model')
        
        # Representation Function
        h_input = tf.keras.Input(shape=(192))
        h = tf.keras.layers.Dense(96, activation=tf.nn.elu)(h_input)
        h = tf.keras.layers.Dense(96, activation=tf.nn.elu)(h)
        h = tf.keras.layers.Dense(96, activation=tf.nn.elu)(h)
        self.Representation = tf.keras.Model(h_input, h, name='Representation Model')
        
        # Action encoder (take in embedded_units and policy)
        e_input = tf.keras.Input(shape=(23))
        e = tf.keras.layers.Dense(32, activation=tf.nn.elu)(e_input)
        e = tf.reduce_sum(e, axis=0, keepdims=True)
        e = tf.keras.layers.Dense(32, activation=tf.nn.elu)(e)
        e = tf.keras.layers.Dense(32, activation=tf.nn.softmax)(e)
        self.policyEncoder = tf.keras.Model(e_input, e, name='Policy Encoder Model')
        
        # Action decoder for each type of units
        d_worker_input = tf.keras.Input(shape=(28))
        d_worker = tf.keras.layers.Dense(16, activation=tf.nn.elu)(d_worker_input)
        d_worker = tf.keras.layers.Dense(7, activation=tf.nn.softmax)(d_worker)
        self.policyDecoderWorker = tf.keras.Model(d_worker_input, d_worker, name='Worker Policy Decoder Model')
        
        d_cart_input = tf.keras.Input(shape=(26))
        d_cart = tf.keras.layers.Dense(16, activation=tf.nn.elu)(d_cart_input)
        d_cart = tf.keras.layers.Dense(5, activation=tf.nn.softmax)(d_cart)
        self.policyDecoderCart = tf.keras.Model(d_cart_input, d_cart, name='Cart Policy Decoder Model')
        
        d_city_input = tf.keras.Input(shape=(26))
        d_city = tf.keras.layers.Dense(16, activation=tf.nn.elu)(d_city_input)
        d_city = tf.keras.layers.Dense(3, activation=tf.nn.softmax)(d_city)
        self.policyDecoderCity = tf.keras.Model(d_city_input, d_city, name='City Policy Decoder Model')
        
    
    def representation(self, game_state):
        # Preprocessing from game_state
        unit_embedding, embedded_units, raw_units = self.unitEncoder(game_state)
        map_embedding, embedded_map = self.spatialEncoder(game_state, embedded_units, raw_units)
        stats_embedding = self.scalarEncoder(game_state)
        
        x = tf.concat([unit_embedding, map_embedding, stats_embedding], axis=1)
        
        # Passing through representation model
        if self.prev_hidden_state is not None:
            x = tf.concat([x, self.prev_hidden_state], axis=-1)
            hidden_state = self.Representation(x)
        else:
            x = tf.concat([x, x], axis=-1)
            self.prev_hidden_state = hidden_state = self.Representation(x)
        
        return hidden_state, embedded_units, raw_units
    
    
    def predict(self, hidden_state):
        encoded_policies, value = self.Prediction(hidden_state)
        return encoded_policies, value
   

    def dynamic(self, hidden_state, encoded_policies):
        predicted_hidden_state, reward = self.Dynamic([hidden_state, encoded_policies])
        return predicted_hidden_state, reward
    
    
    def search(self, hidden_state, encoded_policies, value):
        tree = MCTS(hidden_state, encoded_policies, value)
        for s in range(self.simulation_limit):
            # Selection based on UCB score
            path, selected_policy_index = tree.select()
            node = path[-1]
            
            # Expansion using Dynamic model and prediction model
            node_hidden_state = node.state
            node_selected_policy = node.P.numpy()                   # This is really ugly eagertensor value assignment
            node_selected_policy[selected_policy_index] = 1.0
            node_selected_policy = tf.constant([node_selected_policy])
            next_node_hidden_state, node_reward = self.Dynamic([node_hidden_state, node_selected_policy])
            next_policy, next_value = self.Prediction(next_node_hidden_state)
            node.children[selected_policy_index] = Node(next_node_hidden_state, next_policy, node_reward)
            path.append(node.children[selected_policy_index])
            
            # Backup 
            for k in range(len(path)):
                G = next_value * self.discount_factor ** (len(path)-k) + sum([node.R * self.discount_factor**(len(path)-k) for i in range(k, len(path))])
                G = G.numpy()[0,0] if G.ndim == 2 else G
                path[k].Q = ((path[k].visit_count * path[k].Q + G) / (path[k].visit_count + 1))
                path[k].visit_count += 1
            
            # TODO: normalize all Q value to [0, 1] range
        sampled_policy = tf.constant([[node.visit_count if node is not None else 0 for node in tree.root.children]], dtype=tf.float32)
        sampled_policy = tf.nn.softmax(sampled_policy)
        return sampled_policy, tree
    
    
    def get_actions(self, game_state, encoded_policies, embedded_units, raw_units, player):
        
        padding = (32 - game_state.map_height) // 2
        # Decoding policy
        policies = []
        actions = []
        anything = []
        for i in range(raw_units.shape[-2]):
            if bits_to_dec(raw_units[..., i, 13:17]) > 0:
                continue
            if raw_units[..., i, 12] == 1.0:
                city_input = tf.concat([encoded_policies[..., 12:22], embedded_units[..., i, :]], axis=-1)
                policy = self.policyDecoderCity(city_input)[0]
                
                # Search for the target city tile
                target_city_tile = None
                for city in player.cities.values():
                    for city_tile in city.citytiles:
                        if city_tile.pos.x + padding == bits_to_dec(raw_units[..., i, 1:6]) and city_tile.pos.y + padding == bits_to_dec(raw_units[..., i, 7:12]):
                            target_city_tile = city_tile
                            break
                if target_city_tile is None:
                    continue
                    
                # Translate to action
                if tf.argmax(policy) == 0:
                    actions.append(target_city_tile.research())
                elif tf.argmax(policy) == 1:
                    actions.append(target_city_tile.build_worker())
                elif tf.argmax(policy) == 2:
                    actions.append(target_city_tile.build_cart())
                    
            elif raw_units[..., i, 6] == 1.0:
                worker_input = tf.concat([encoded_policies[..., :12], embedded_units[..., i, :]], axis=-1)
                policy = self.policyDecoderWorker(worker_input)[0]
                
                # Search for the target unit
                target_unit = None
                for unit in player.units:
                    if unit.pos.x + padding == bits_to_dec(raw_units[..., i, 1:6]) and unit.pos.y + padding == bits_to_dec(raw_units[..., i, 7:12]) and unit.cooldown == bits_to_dec(raw_units[..., i, 13:17]) and unit.is_worker():
                        target_unit = unit
                        break 
                if target_unit is None:
                    continue
                
                # Translate to action
                if tf.argmax(policy) == 0:
                    actions.append(target_unit.move('c'))
                elif tf.argmax(policy) == 1:
                    actions.append(target_unit.move('n'))
                elif tf.argmax(policy) == 2:
                    actions.append(target_unit.move('e'))
                elif tf.argmax(policy) == 3:
                    actions.append(target_unit.move('s'))
                elif tf.argmax(policy) == 4:
                    actions.append(target_unit.move('w'))
                elif tf.argmax(policy) == 5:
                    actions.append(target_unit.build_city())
                elif tf.argmax(policy) == 6:
                    actions.append(target_unit.pillage())
            else:
                cart_input = tf.concat([encoded_policies[..., 22:], embedded_units[..., i, :]], axis=-1)
                policy = self.policyDecoderCart(cart_input)[0]
                
                # Search for the target unit
                target_unit = None
                for unit in player.units:
                    if unit.pos.x + padding == bits_to_dec(raw_units[..., i, 1:6]) and unit.pos.y + padding == bits_to_dec(raw_units[..., i, 7:12]) and unit.cooldown == bits_to_dec(raw_units[..., i, 13:17]) and unit.is_cart:
                        target_unit = unit
                        break 
                if target_unit is None:
                    continue
                
                # Translate to action
                if tf.argmax(policy) == 0:
                    actions.append(target_unit.move('c'))
                elif tf.argmax(policy) == 1:
                    actions.append(target_unit.move('n'))
                elif tf.argmax(policy) == 2:
                    actions.append(target_unit.move('e'))
                elif tf.argmax(policy) == 3:
                    actions.append(target_unit.move('s'))
                elif tf.argmax(policy) == 4:
                    actions.append(target_unit.move('w'))
            policies.append(policy)

        return actions, policies
    
    
    def replay(self, batch_size):
        minibatch = random.sample(self.memory, batch_size)
        
        policies = []
        values = []
        target_policies = []
        target_values = []
        
        # print(minibatch)
        
        for game in minibatch:
            index = random.randint(0, len(game)-self.unroll_step)
            for experience in game[index-1:index+self.unroll_step-1]:
                loss = 0
                game_state, tree = experience
                
                # target_policy = sampled_policy
                # target_value = max([node.Q if node is not None for node in tree.root.children])
                # target_value = tree.root.Q
                
                with tf.GradientTape() as tape:
                    hidden_state, embedded_units, raw_units = self.representation(game_state)
                    encoded_policies, value = self.predict(hidden_state)
                    next_hidden_state, reward = self.dynamic(hidden_state, encoded_policies) 
                    sampled_policy, tree = self.search(hidden_state, encoded_policies, value)
                    actions, policies = self.get_actions(game_state, sampled_policy, embedded_units, raw_units, player)
                
                loss = self.loss_cce(tree.root.P, encoded_policies[0]) + self.loss_mse(tf.constant([tree.root.Q]), value)
                gradients = tape.gradient(loss, tape.watched_variables())
                self.optimizer.apply_gradients(
                    (grad, var)
                    for (grad, var) in zip(gradients, tape.watched_variables())
                    if grad is not None)
                
                print(f"Loss: {loss}")
                
#                 print(encoded_policies)
#                 print(tree.root.P)
#                 print(value)
#                 print(tf.constant([tree.root.Q]))
                
    
    def save(self):
        self.memory.append(self.buffer)
        print(f"Memory size: {len(self.memory)}")
        self.buffer = []
    
                                       
    def __call__(self, game_state, player=None):
        if player is None:
            player = game_state.players[0]
        
        hidden_state, embedded_units, raw_units = self.representation(game_state)
        encoded_policies, value = self.predict(hidden_state)
        next_hidden_state, reward = self.dynamic(hidden_state, encoded_policies) 
        sampled_policy, tree = self.search(hidden_state, encoded_policies, value)
        actions, policies = self.get_actions(game_state, sampled_policy, embedded_units, raw_units, player)
        
        experience = (copy.deepcopy(game_state), tree)
        self.buffer.append(experience)
        
        return actions, experience

In [ ]:
# Delux Agent Test
agent = Delux()
actions, experience = agent(game_state)
# actions, experience

## Replay Buffer

## Data Generation and Training

In [ ]:
episode = 100

agent = Delux(simulation_limit=100)
game_state = Game()
env = make("lux_ai_2021", configuration={"seed": 4242, "loglevel": 0}, debug=True)
trainer = env.train([None, "simple_agent"])

for i in range(1, episode+1):
    env.reset()
    obs = trainer.reset()
    game_state._initialize(env.state[0].observation["updates"])
    game_state._update(env.state[0].observation["updates"][2:])
    
    player = game_state.players[env.state[0].observation.player]
    opponent = game_state.players[(env.state[1].observation.player + 1) % 2]
    width, height = game_state.map.width, game_state.map.height
    
    for episode in tqdm(range(360)):
        actions, experience = agent(game_state, player)

        obs, _, done, _ = trainer.step(actions) # obs, reward, done, info
        
        game_state._update(obs["updates"])
        player = game_state.players[env.state[0].observation.player]
        opponent = game_state.players[(env.state[1].observation.player + 1) % 2]
    
        if done:
            
            player_score = game_state.players[0].city_tile_count * 3 + len(game_state.players[0].units) + game_state.players[0].research_points / 100
            opponent_score = game_state.players[1].city_tile_count * 3 + len(game_state.players[1].units) + game_state.players[1].research_points / 100
            
            print(f"Episode {i}\nPlayer score: {player_score}\nOpponent score: {opponent_score}\nSurvived turn: {obs.step}")
            
            agent.save()
            break
        
    if i % 5 == 0:
        agent.replay(batch_size=5)

In [ ]:
# a = Delux()
# b = Delux(is_opponent=True)

game_state = None
def trained_agent(observation, configuration):
    global game_state

    ### Do not edit ###
    if observation["step"] == 0:
        game_state = Game()
        game_state._initialize(observation["updates"])
        game_state._update(observation["updates"][2:])
        game_state.id = observation.player
    else:
        game_state._update(observation["updates"])
    
    actions = []

    ### AI Code goes down here! ### 
    player = game_state.players[observation.player]
    opponent = game_state.players[(observation.player + 1) % 2]
    width, height = game_state.map.width, game_state.map.height
    
    actions = agent(game_state, player)
    print(actions)
    return actions

# def agent2(observation, configuration):
#     global game_state

#     ### Do not edit ###
#     if observation["step"] == 0:
#         game_state = Game()
#         game_state._initialize(observation["updates"])
#         game_state._update(observation["updates"][2:])
#         game_state.id = observation.player
#     else:
#         game_state._update(observation["updates"])
    
#     actions = []

#     ### AI Code goes down here! ### 
#     player = game_state.players[observation.player]
#     opponent = game_state.players[(observation.player + 1) % 2]
#     width, height = game_state.map.width, game_state.map.height
    
#     actions = b(game_state, player)
#     print(actions)
#     return actions


env = make("lux_ai_2021", configuration={"seed": np.random.randint(10000000), "loglevel": 2}, debug=True)
steps = env.run([trained_agent, "simple_agent"])
env.render(mode="ipython", width=1200, height=800)

In [ ]:
env.render(mode="ipython", width=1200, height=800)